# Qwen Math generation under different chat templates

Examine how the **chat template** (Qwen's native template vs. the
vendored Llama-3.1 `custom_chat_template`) affects what
Qwen2.5-Math-1.5B-Instruct actually *generates*, in two settings:

1. **Fresh generation** — start the assistant turn from the
   question (`add_generation_prompt=True`).
2. **Continuation** — continue an in-progress assistant turn
   after some existing steps (`continue_final_message=True` +
   the strip-and-reappend of the `\n\n` step separator), exactly
   as the search loop expands a node.

This is the *generation* counterpart to the tokenizer-only
`examine_llm_chat_templates_v1` (which only renders prompts) and
`test_step_separator_affect_generation` (which checks P(EOS) at
one position). Here we drive vLLM end-to-end to see the text.
See `examine_llm_generation_templates_llama_v1` for the Llama
counterpart.

For Qwen the native template is Qwen's own `<|im_start|>` format
(no BOS, no date preamble). The custom template forces Qwen into
Llama-3.1 format — off-distribution for Qwen (see
`examine_llm_chat_templates_v1`). Watch whether that degrades the
generation vs. Qwen's native template, in both settings.

Backend: **vLLM** (matches the BoN / MCTS pipeline). fp16 on the
V100 (sm_70, no bf16).

Why manual templating: to control native vs. custom and the
continuation path we apply the chat template with the HF
tokenizer and feed raw strings to `llm.generate()` — the same
thing the search code does — rather than `llm.chat()` (which
would template internally with the model's native template).

## Setup

In [1]:
import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL+1)
logging.disable(logging.CRITICAL)

import gc
import sys
sys.path.append("..")

import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

from notebook_utils import gpu_mem_used_gb
from utils.configs import (
    build_conv, system_prompt, custom_chat_template,
)

# Model paths
base_dir = "/groups/chichengz/tnn/datasets"
qwen_dir = f"{base_dir}/Qwen2.5-Math-7B-Instruct"

## Helpers

In [2]:
STEP_SEP = "\n\n"

sal_system_prompt = system_prompt  # sal Llama prompt from utils.configs
qwen_math_system_prompt = (
    "Please reason step by step, and put your final answer "
    "within \\boxed{}."
)


def make_prompt(tokenizer, setting, sys_prompt, date_string=None):
    """Template a prompt for the given setting using the tokenizer's
    currently-set chat_template.

    setting == "fresh":  start the assistant turn from the question
        (add_generation_prompt=True, empty assistant response).
    setting == "continue": continue an in-progress assistant turn
        after current_text, using the strip-and-reappend pattern
        the search code uses to preserve the trailing separator.
    """
    kwargs = {"tokenize": False}
    if date_string is not None:
        kwargs["date_string"] = date_string
    if setting == "fresh":
        convs = [build_conv(question, "", sys_prompt)]
        return tokenizer.apply_chat_template(
            convs, add_generation_prompt=True, **kwargs,
        )[0]
    if setting == "continue":
        clean = current_text.removesuffix(STEP_SEP)
        convs = [build_conv(question, clean, sys_prompt)]
        templated = tokenizer.apply_chat_template(
            convs,
            add_generation_prompt=False,
            continue_final_message=True,
            **kwargs,
        )[0]
        # Re-append the separator stripped before templating.
        return templated + STEP_SEP
    raise ValueError(setting)


def show_generation(llm, tokenizer, sampling_params, sys_prompt,
                    date_string=None):
    """For each (template, setting), template a prompt and generate
    one continuation; print the full prompt and the output."""
    native = tokenizer.chat_template
    for tmpl_name, tmpl in [("native", native),
                             ("custom", custom_chat_template)]:
        tokenizer.chat_template = tmpl
        for setting in ("fresh", "continue"):
            prompt = make_prompt(
                tokenizer, setting, sys_prompt, date_string,
            )
            out = llm.generate(
                prompt, sampling_params, use_tqdm=False,
            )[0].outputs[0]
            print(f"===== template={tmpl_name}  setting={setting} =====")
            print(f"prompt ends with sep: {prompt.endswith(STEP_SEP)}")
            print("--- full prompt ---")
            print(prompt)
            print("--- generated ---")
            print(out.text)
            print(f"[stop_reason={out.stop_reason!r} "
                  f"ntok={len(out.token_ids)}]")
            print()
    tokenizer.chat_template = native  # restore

## Load LLM

In [3]:
llm = LLM(
    model=qwen_dir,
    dtype="float16",            # V100 (sm_70): no bf16
    max_model_len=4096,         # Math-1.5B max_position_embeddings=4096
    gpu_memory_utilization=0.5,
    enforce_eager=True,
    seed=0,
)
tokenizer = AutoTokenizer.from_pretrained(qwen_dir)
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu126).
Cannot use FA version 2 is not supported due to FA2 is only supported on devices with compute capability >= 8
Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  25% Completed | 1/4 [00:02<00:06,  2.13s/it]
Loading safetensors checkpoint shards:  50% Completed | 2/4 [00:04<00:04,  2.18s/it]
Loading safetensors checkpoint shards:  75% Completed | 3/4 [00:06<00:02,  2.18s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:08<00:00,  2.11s/it]
Loading safetensors checkpoint shards: 100% Completed | 4/4 [00:08<00:00,  2.13s/it]

Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ign

GPU memory used: 16.12 GB


## Sampling parameters

In [4]:
# Match the search loop: generate one step, stopping at the next
# separator (kept in the output so we can see it).
sampling_params = SamplingParams(
    temperature=0.8,
    top_p=1.0,
    max_tokens=512,
    stop=[STEP_SEP],
    include_stop_str_in_output=True,
    seed=0,
)

## Example

One real prm800k-style question. For the continuation setting we
supply two already-written steps (`current_text`, `\n\n`-joined,
trailing separator) and ask the model for **Step 3**.

In [5]:
question = (
    "The set of points $(x,y,z)$ that satisfy\n\\[2x = 3y = -z\\]"
    "is a line.\n\nThe set of points $(x,y,z)$ that satisfy\n"
    "\\[6x = -y = -4z\\]is another line.\n\nFind the angle "
    "between these lines, in degrees."
)

# Two existing steps for the continuation setting (trailing \n\n).
current_text = (
    "## Step 1: Identify the direction vectors of the lines.\n"
    "For the first line the direction vector is (2, 3, -1); for "
    "the second line it is (6, -1, -4).\n\n"
    "## Step 2: Recall the formula for the angle between vectors.\n"
    "cos(theta) = (a . b) / (|a| |b|).\n\n"
)

In [6]:
show_generation(
    llm, tokenizer, sampling_params,
    sal_system_prompt, date_string="Aug 1 2025",
)

===== template=native  setting=fresh =====
prompt ends with sep: False
--- full prompt ---
<|im_start|>system
Solve the following math problem efficiently and clearly:

- For simple problems (2 steps or fewer):
Provide a concise solution with minimal explanation.

- For complex problems (3 steps or more):
Use this step-by-step format:

## Step 1: [Concise description]
[Brief explanation and calculations]

## Step 2: [Concise description]
[Brief explanation and calculations]

...

Regardless of the approach, always conclude with:

Therefore, the final answer is: $\boxed{answer}$. I hope it is correct.

Where [answer] is just the final number or expression that solves the problem.<|im_end|>
<|im_start|>user
The set of points $(x,y,z)$ that satisfy
\[2x = 3y = -z\]is a line.

The set of points $(x,y,z)$ that satisfy
\[6x = -y = -4z\]is another line.

Find the angle between these lines, in degrees.<|im_end|>
<|im_start|>assistant

--- generated ---
To find the angle between the two lines g

## Example — Qwen math system prompt

Same question and continuation, but with Qwen2.5-Math-Instruct's
recommended system prompt instead of the `sal` Llama prompt.
Tests whether the preamble is driven by the prompt (prediction:
yes — native/fresh should open with `## Step 1` or no preamble).

In [7]:
show_generation(
    llm, tokenizer, sampling_params,
    qwen_math_system_prompt,
)

===== template=native  setting=fresh =====
prompt ends with sep: False
--- full prompt ---
<|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
The set of points $(x,y,z)$ that satisfy
\[2x = 3y = -z\]is a line.

The set of points $(x,y,z)$ that satisfy
\[6x = -y = -4z\]is another line.

Find the angle between these lines, in degrees.<|im_end|>
<|im_start|>assistant

--- generated ---
To find the angle between the two lines given by the equations \(2x = 3y = -z\) and \(6x = -y = -4z\), we first need to determine the direction vectors of these lines.


[stop_reason='\n\n' ntok=48]

===== template=native  setting=continue =====
prompt ends with sep: True
--- full prompt ---
<|im_start|>system
Please reason step by step, and put your final answer within \boxed{}.<|im_end|>
<|im_start|>user
The set of points $(x,y,z)$ that satisfy
\[2x = 3y = -z\]is a line.

The set of points $(x,y,z)$ that satisfy
\[6x = -y = -4z\]is another 

## Cleanup

In [ ]:
del llm, tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory used: {gpu_mem_used_gb():.2f} GB")